# Regularization recipes and synthetic engine proof for v007

Definitions only: execute the unchanged `sequence_components_v005.ipynb` code cells
first, then this notebook's code cells. Defining these helpers reads no assignment
data and starts no fits.

The existing v005 engine **already** supports Adam and AdamW: `_train_configuration`
accepts both, `_optimizer` selects the corresponding Torch class, and the entire
canonical training configuration is included in the exact-resume fingerprint.
Consequently this extension overrides **zero** engine functions, changes **zero**
engine source lines, and retains the v005 checkpoint format. Adam's defaults,
preprocessing, loss, clipping, schedule, model, RNG capture, optimizer-state capture,
and resumption guard remain unchanged. No source strings are rewritten.

`regularization_recipes_v007` preserves the caller's current stacked LSTM package
(including deep widths, residual connections, and output concatenation). It offers
five recipes without executing them: original Adam/zero decay, AdamW/zero decay,
AdamW/1e-4 decay, AdamW/1e-3 decay, and Adam/zero decay with dropout .25. Every arm
uses the original 200 saved rates, same seed, and unchanged loss/scaling/batch size.
Compare decay arms with AdamW/zero decay; compare dropout with the original Adam
recipe. Main orchestration selects which prespecified arms have authorization.

`regularization_engine_proof_v007(device)` uses **synthetic data only**, with a tiny
three-layer bidirectional residual LSTM and explicit dropout. It checks that engine
functions retain their identities, actual optimizer classes and zero-decay control
agreement, a nonzero-decay parameter effect, active .25 dropout, resume rejection
when optimizer/decay/dropout changes, and exact checkpoint/RNG/optimizer continuation
for both validation selection and fixed-schedule fitting. Adam versus AdamW at zero
decay is measured with explicit tolerances, not presumed bitwise identical across
Torch versions or devices. Run this proof on the actual course CUDA environment
before deciding whether its original Adam trajectory is an adequate AdamW/zero
control. A tiny synthetic equivalence result is implementation evidence; it is not
a guarantee that long real-data trajectories coincide.


In [ ]:
# Preserve every v005 function; fail clearly if the required engine is absent.
_V007_REQUIRED_ENGINE = (
    'canonical_model_spec', '_train_configuration', 'deterministic_lr_schedule',
    'build_model', '_optimizer', '_prepare_run', '_fingerprint',
    '_rng_state', '_restore_rng', '_checkpoint_payload',
    'fit_validate', 'fit_fixed', 'predict_checkpoint',
)
_missing_v007 = [name for name in _V007_REQUIRED_ENGINE if name not in globals()]
if _missing_v007:
    raise RuntimeError('Execute unchanged v005 engine code cells first: ' + ', '.join(_missing_v007))
_V007_ORIGINAL_ENGINE = {name: globals()[name] for name in _V007_REQUIRED_ENGINE}
V007_REGULARIZATION_SEED = 20261102
V007_REGULARIZATION_ARMS = (
    'original_adam', 'adamw_zero', 'adamw_1e4', 'adamw_1e3', 'dropout_025',
)


def regularization_recipes_v007(base_model_spec, base_train_cfg,
                                 frozen_lr_schedule=None, seed=V007_REGULARIZATION_SEED):
    """Return five independent recipes without reading data or fitting a model.

    The caller supplies the approved baseline package and its saved 200 rates.
    Configuration is canonicalized with the unchanged v005 validators. No new
    model seed, warmup, LR shape, architecture component, or loss is introduced.
    """
    spec = canonical_model_spec(base_model_spec)
    cfg = _train_configuration(base_train_cfg)
    if spec['family'] != 'lstm' or spec['layers'] < 2 or spec['dropout'] != .15:
        raise ValueError('Expected the current stacked LSTM baseline with dropout .15')
    if cfg['optimizer'] != 'Adam' or cfg['weight_decay'] != 0.:
        raise ValueError('Expected the original Adam/zero-decay baseline')
    if cfg['max_epochs'] != 200 or cfg['scheduler'] not in ('replay', 'fixed'):
        raise ValueError('Expected the original 200-epoch replay/fixed configuration')
    if int(seed) != V007_REGULARIZATION_SEED:
        raise ValueError('v007 preserves the original model seed 20261102')
    rates = list(cfg['frozen_lr_schedule'])
    if frozen_lr_schedule is not None and list(frozen_lr_schedule) != rates:
        raise ValueError('Supplied rates differ from the original frozen schedule')

    definitions = (
        ('original_adam', 'Adam', 0., .15, None, 'unchanged original baseline'),
        ('adamw_zero', 'AdamW', 0., .15, 'original_adam', 'optimizer implementation control'),
        ('adamw_1e4', 'AdamW', 1e-4, .15, 'adamw_zero', 'decoupled weight decay only'),
        ('adamw_1e3', 'AdamW', 1e-3, .15, 'adamw_zero', 'decoupled weight decay only'),
        ('dropout_025', 'Adam', 0., .25, 'original_adam', 'inter-layer dropout only'),
    )
    arms = {}
    for name, optimizer, decay, dropout, reference, contrast in definitions:
        arm_spec = copy.deepcopy(spec)
        arm_cfg = copy.deepcopy(cfg)
        arm_spec['dropout'] = dropout
        arm_cfg.update(optimizer=optimizer, weight_decay=decay)
        arms[name] = dict(model_spec=arm_spec, train_cfg=arm_cfg,
            lr_schedule=list(rates), seed=int(seed), comparison_reference=reference,
            contrast=contrast)
    assert tuple(arms) == V007_REGULARIZATION_ARMS
    assert arms['original_adam']['model_spec'] == spec
    assert arms['original_adam']['train_cfg'] == cfg
    assert {key for key in spec if spec[key] != arms['dropout_025']['model_spec'][key]} == {'dropout'}
    for name in ('adamw_1e4', 'adamw_1e3'):
        assert {key for key in cfg if arms['adamw_zero']['train_cfg'][key] != arms[name]['train_cfg'][key]} == {'weight_decay'}
    return arms


In [ ]:
# Optional synthetic proof. This function is never called while defining cells.
def regularization_engine_proof_v007(device='cpu', output_dir=None):
    """Run small synthetic checks on this Torch/device; return JSON-safe evidence.

    Temporary checkpoints are removed on completion. No assignment data paths,
    dataset reads, production checkpoints, or approved-data fits are involved.
    Timing fields and checkpoint file bytes are not exact-continuation targets;
    model/optimizer/preprocessing/RNG states and numerical epoch records are.
    """
    import platform
    import tempfile

    if not all(globals()[name] is function for name, function in _V007_ORIGINAL_ENGINE.items()):
        raise AssertionError('A protected v005 engine function was replaced')
    target_device = _sequence_device(device)
    rng = np.random.default_rng(7007)
    x = rng.normal(size=(8, 80, 5)).astype(np.float32)
    x[..., 0] = np.asarray([5, 20, 50, 5, 20, 50, 5, 50], dtype=np.float32)[:, None]
    x[..., 1] = np.asarray([10, 20, 50, 20, 50, 10, 50, 20], dtype=np.float32)[:, None]
    x[..., 2] = np.arange(80, dtype=np.float32)[None, :] / 40.
    out = np.broadcast_to((np.arange(80) >= 53).astype(np.int8), (8, 80)).copy()
    y = (2. + .3*x[..., 3] + .05*x[..., 4] + .01*x[..., 0]
         + .2*np.sin(x[..., 2])).astype(np.float32)
    spec = canonical_model_spec(dict(family='lstm', hidden=6, layers=3,
        dropout=.15, bidirectional=True, widths=[6, 5, 4], residual=True,
        output_concat=True, head_width=5))
    rates = [1e-3, 7e-4, 2e-4]
    cfg = _train_configuration(dict(lr=1e-3, batch_size=2, max_epochs=3,
        scheduler='replay', frozen_lr_schedule=rates, prediction_batch_size=2,
        patience=0, optimizer='Adam', weight_decay=0.))
    seed = V007_REGULARIZATION_SEED
    feature_names = ['R', 'C', 'time_step', 'synthetic_control', 'synthetic_aux']
    tolerances = dict(parameter_atol=1e-7, parameter_rtol=1e-6,
                      prediction_atol=1e-6, prediction_rtol=1e-6)

    def exact(left, right):
        if torch.is_tensor(left) and torch.is_tensor(right):
            return left.dtype == right.dtype and left.shape == right.shape and torch.equal(left, right)
        if isinstance(left, np.ndarray) and isinstance(right, np.ndarray):
            return left.dtype == right.dtype and np.array_equal(left, right)
        if isinstance(left, dict) and isinstance(right, dict):
            return left.keys() == right.keys() and all(exact(left[key], right[key]) for key in left)
        if isinstance(left, (list, tuple)) and isinstance(right, type(left)):
            return len(left) == len(right) and all(exact(a, b) for a, b in zip(left, right))
        return type(left) is type(right) and left == right

    def payload(path):
        return torch.load(path, map_location='cpu', weights_only=False)

    def epoch_numerics(history):
        return [{key: value for key, value in record.items()
                 if key not in ('epoch_seconds', 'elapsed_seconds')} for record in history]

    def matching_checkpoints(left, right):
        a, b = payload(left), payload(right)
        keys = ('state_dict', 'optimizer_state', 'rng_state', 'preprocess',
                'model_spec', 'config', 'fingerprint', 'epoch', 'optimizer_steps',
                'completed', 'termination', 'scheduler_state')
        checks = {key: exact(a[key], b[key]) for key in keys}
        checks['epoch_numerics'] = exact(epoch_numerics(a['epochs']), epoch_numerics(b['epochs']))
        if 'best_epoch' in a or 'best_epoch' in b:
            checks['best_selection'] = (a.get('best_epoch') == b.get('best_epoch')
                and a.get('best_val_mae') == b.get('best_val_mae'))
        assert all(checks.values()), checks
        return checks

    def validation(path, train_cfg, model_spec=spec, resume=False, on_epoch=None, fit=None):
        return (fit or fit_validate)(x[:6], y[:6], out[:6], x[6:], y[6:], out[6:],
            model_spec, train_cfg, seed, path, feature_names=feature_names,
            train_diagnostic_breaths=6, device=str(target_device), resume=resume,
            on_epoch=on_epoch)

    def fixed(path, train_cfg, resume=False, on_epoch=None):
        return fit_fixed(x[:6], y[:6], out[:6], spec, train_cfg, seed, rates,
            path, feature_names=feature_names, device=str(target_device),
            resume=resume, on_epoch=on_epoch)

    class ProofInterruption(RuntimeError):
        pass

    def interrupt_after_first(record):
        if record['epoch'] == 1:
            raise ProofInterruption('Synthetic interruption after authoritative checkpoint write')

    # Pure configuration proof of the actual deeper package; no large model fit.
    deep_spec = canonical_model_spec(dict(family='lstm', hidden=256, layers=3,
        widths=[256, 192, 128], dropout=.15, bidirectional=True, residual=True,
        output_concat=True, head_width=32))
    cosine_cfg = dict(cfg, max_epochs=200, scheduler='cosine', warmup_epochs=5,
                      frozen_lr_schedule=None)
    frozen200 = deterministic_lr_schedule(cosine_cfg)
    deep_cfg = dict(cfg, max_epochs=200, frozen_lr_schedule=frozen200)
    recipes = regularization_recipes_v007(deep_spec, deep_cfg, frozen200)
    assert recipes['original_adam']['model_spec'] == deep_spec
    assert recipes['original_adam']['train_cfg'] == deep_cfg
    assert all(arm['lr_schedule'] == frozen200 and arm['seed'] == seed for arm in recipes.values())
    for name in ('adamw_zero', 'adamw_1e4', 'adamw_1e3'):
        assert recipes[name]['model_spec'] == deep_spec
    dropout_model_changes = {key for key in deep_spec
        if recipes['dropout_025']['model_spec'][key] != deep_spec[key]}
    assert dropout_model_changes == {'dropout'}
    assert recipes['dropout_025']['train_cfg'] == deep_cfg
    recipe_checks = dict(arms=list(recipes), epoch_count=200, seed=seed,
        original_recipe_unchanged=True, original_frozen_rates_preserved=True,
        deep_architecture_preserved=True, independent_decay_and_dropout=True,
        dropout_model_changes=sorted(dropout_model_changes),
        decay_comparison_reference='adamw_zero', dropout_comparison_reference='original_adam')

    if output_dir is not None:
        Path(output_dir).mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='regularization-v007-proof-', dir=output_dir) as temporary:
        directory = Path(temporary)
        adam_path, repeat_path, adamw_path = (directory / name for name in ('adam.pt', 'adam_repeat.pt', 'adamw_zero.pt'))
        original_report = validation(adam_path, cfg, fit=_V007_ORIGINAL_ENGINE['fit_validate'])
        repeat_report = validation(repeat_path, cfg)
        baseline_checks = matching_checkpoints(adam_path, repeat_path)
        baseline_last_checks = matching_checkpoints(_last_path(adam_path), _last_path(repeat_path))
        assert original_report['validation_mae'] == repeat_report['validation_mae']
        adamw_cfg = dict(cfg, optimizer='AdamW')
        adamw_report = validation(adamw_path, adamw_cfg)
        a, b = payload(_last_path(adam_path)), payload(_last_path(adamw_path))
        parameter_max_difference = max(float((a['state_dict'][key] - b['state_dict'][key]).abs().max())
                                       for key in a['state_dict'])
        parameter_close = all(torch.allclose(a['state_dict'][key], b['state_dict'][key],
            atol=tolerances['parameter_atol'], rtol=tolerances['parameter_rtol']) for key in a['state_dict'])
        adam_prediction = predict_checkpoint(adam_path, x[6:], device=str(target_device), batch_size=2)
        adamw_prediction = predict_checkpoint(adamw_path, x[6:], device=str(target_device), batch_size=2)
        prediction_difference = float(np.max(np.abs(adam_prediction-adamw_prediction)))
        prediction_close = bool(np.allclose(adam_prediction, adamw_prediction,
            atol=tolerances['prediction_atol'], rtol=tolerances['prediction_rtol']))
        optimizer_classes = {}
        for optimizer_name in ('Adam', 'AdamW'):
            optimizer = _optimizer(build_model(spec, 5), dict(cfg, optimizer=optimizer_name))
            optimizer_classes[optimizer_name] = type(optimizer).__name__
            assert type(optimizer) is getattr(torch.optim, optimizer_name)
            assert optimizer.param_groups[0]['weight_decay'] == 0.
        decay_path = directory / 'adamw_decay.pt'
        decay_cfg = dict(adamw_cfg, weight_decay=1e-3)
        validation(decay_path, decay_cfg)
        decayed = payload(_last_path(decay_path))
        decay_parameter_difference = max(float((b['state_dict'][key]-decayed['state_dict'][key]).abs().max())
                                         for key in b['state_dict'])
        assert decay_parameter_difference > 0.
        assert decayed['config']['weight_decay'] == 1e-3
        assert all(group['weight_decay'] == 1e-3 for group in decayed['optimizer_state']['param_groups'])

        dropout_spec = dict(spec, dropout=.25)
        seed_sequence(seed)
        dropout_model = build_model(dropout_spec, 5).to(target_device)
        assert dropout_model.explicit_stack and dropout_model.between_layers.p == .25
        sample = torch.ones((2, 80, 6), device=target_device); sample[..., -1] = 0.
        dropout_model.train()
        with torch.no_grad():
            train_a, train_b = dropout_model(sample), dropout_model(sample)
        dropout_model.eval()
        with torch.no_grad():
            eval_a, eval_b = dropout_model(sample), dropout_model(sample)
        assert not torch.equal(train_a, train_b)
        assert torch.equal(eval_a, eval_b)

        validate_resume_path = directory / 'adamw_validate_resume.pt'
        try:
            validation(validate_resume_path, decay_cfg, on_epoch=interrupt_after_first)
        except ProofInterruption:
            pass
        else:
            raise AssertionError('Synthetic interruption did not occur')
        rejected = {}
        for name, altered_cfg, altered_spec in (
            ('optimizer', dict(decay_cfg, optimizer='Adam'), spec),
            ('weight_decay', dict(decay_cfg, weight_decay=0.), spec),
            ('dropout', decay_cfg, dropout_spec),
        ):
            try:
                validation(validate_resume_path, altered_cfg, model_spec=altered_spec, resume=True)
            except ValueError as error:
                rejected[name] = 'Resume rejected:' in str(error)
            else:
                rejected[name] = False
        assert all(rejected.values()), rejected
        resumed_validation = validation(validate_resume_path, decay_cfg, resume=True)
        validation_best_checks = matching_checkpoints(decay_path, validate_resume_path)
        validation_last_checks = matching_checkpoints(_last_path(decay_path), _last_path(validate_resume_path))
        assert resumed_validation['validation_mae'] == payload(decay_path)['best_val_mae']
        fixed_path, fixed_resume_path = directory/'adamw_fixed.pt', directory/'adamw_fixed_resume.pt'
        fixed(fixed_path, decay_cfg)
        try:
            fixed(fixed_resume_path, decay_cfg, on_epoch=interrupt_after_first)
        except ProofInterruption:
            pass
        else:
            raise AssertionError('Synthetic fixed-fit interruption did not occur')
        fixed(fixed_resume_path, decay_cfg, resume=True)
        fixed_checks = matching_checkpoints(fixed_path, fixed_resume_path)
        fixed_last_checks = matching_checkpoints(_last_path(fixed_path), _last_path(fixed_resume_path))

    return dict(schema='regularization_engine_proof_v007', scope='synthetic only; no assignment-data fits',
        python_version=platform.python_version(), torch_version=str(torch.__version__),
        numpy_version=str(np.__version__), device=str(target_device),
        cuda_version=torch.version.cuda,
        device_name=torch.cuda.get_device_name(target_device) if target_device.type == 'cuda' else platform.machine(),
        engine_function_overrides=0, engine_function_identities_unchanged=True,
        baseline_original_vs_extension=dict(best_checkpoint=baseline_checks, last_checkpoint=baseline_last_checks),
        optimizer_classes=optimizer_classes, recipe_checks=recipe_checks,
        adamw_zero_vs_adam_zero=dict(tolerances=tolerances, parameter_allclose=bool(parameter_close),
            prediction_allclose=prediction_close, adequate_synthetic_zero_decay_control=bool(parameter_close and prediction_close),
            parameter_max_abs_difference=parameter_max_difference,
            prediction_max_abs_difference=prediction_difference,
            adam_validation_mae=original_report['validation_mae'], adamw_validation_mae=adamw_report['validation_mae']),
        adamw_nonzero_decay=dict(weight_decay=1e-3, parameter_max_abs_difference=decay_parameter_difference,
            distinct_from_adamw_zero=True, checkpoint_and_optimizer_record_decay=True),
        dropout_025=dict(explicit_stack=True, probability=.25, stochastic_in_train=True,
            deterministic_in_eval=True), changed_recipe_resume_rejected=rejected,
        exact_adamw_resume=dict(validation_best=validation_best_checks, validation_last=validation_last_checks,
            fixed=fixed_checks, fixed_last=fixed_last_checks),
        synthetic_shape=list(x.shape), synthetic_model_spec=spec, synthetic_train_cfg=cfg,
        temporary_checkpoints_removed=True)
